# Notebook 7 — Multimodal LoRA (Colab A100)

**Purpose:** The last new experiment type. Tests whether adding
architecture-aware `target_modules` -- covering BOTH BLIP's visual
encoder and text decoder -- fixes the silent zero-visual-parameter
failure documented for the paper's existing "LoRA (text-only)" baseline
(Section VII-C of the paper).

**CRITICAL -- run the diagnostic cell (Cell 8) FIRST**, before any
training. It inspects the actual loaded model's module names and
confirms which `target_modules` pattern will actually match visual-encoder
layers. Do not skip this -- the original LoRA baseline's failure was
caused by exactly this kind of unverified assumption about module names.

**Environment pins:** transformers (Python-3.13-compatible range, per
Notebook 6), peft==0.11.1.

**Structure:** shared setup, diagnostic cell, then Parts A/B/C
(UICD/RSICD/ROCOv2), same pattern as Notebook 6.

**Output:** `multimodal_lora_{dataset}_seed{42,0,123}.json`, including
logged counts of trainable visual vs.\ language parameters -- this is
the number that proves (or disproves) whether the fix worked.

## 1. Mount Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2. Install dependencies

In [ ]:
import subprocess, sys

def pip_install(pkg):
    subprocess.run(
        [sys.executable, '-m', 'pip', 'install', pkg,
         '--no-warn-script-location'],
        check=True
    )

# Same version range as Notebook 6 (RMS-Balanced FT) -- Colab's Python
# 3.13 runtime has no wheel for the tokenizers version transformers==4.41.2
# requires, so we use a compatible newer range instead.
pip_install('transformers>=4.44,<4.50')
pip_install('peft==0.11.1')
pip_install('pycocoevalcap')
pip_install('nltk')
pip_install('datasets')
pip_install('evaluate')
pip_install('kaggle')

print('Installations complete. RESTART RUNTIME NOW, then re-run cells 1-2.')

## IMPORTANT -- restart runtime once, then verify

In [ ]:
import transformers, peft
print(f'transformers: {transformers.__version__}')
print(f'peft: {peft.__version__}')
assert peft.__version__ == '0.11.1', f'STOP: peft is {peft.__version__}. Restart runtime.'
major, minor = (int(x) for x in transformers.__version__.split('.')[:2])
assert (major, minor) >= (4, 44), f'STOP: transformers is {transformers.__version__}, too old.'
print('Versions accepted.')

import nltk
nltk.download('punkt', quiet=True)
nltk.download('punkt_tab', quiet=True)
nltk.download('wordnet', quiet=True)
print('NLTK data downloaded.')

## 3. Imports, config, RAM/disk check before anything else

In [ ]:
import sys, os, gc, json, math, random, shutil as _shutil
from pathlib import Path
import numpy as np
import torch
import psutil

from torch.utils.data import Dataset, DataLoader
from torch.optim import AdamW
from PIL import Image
from collections import defaultdict
from datasets import load_dataset

from transformers import BlipProcessor, BlipForConditionalGeneration
from peft import LoraConfig, get_peft_model
from nltk.translate.bleu_score import corpus_bleu, SmoothingFunction

mem = psutil.virtual_memory()
print(f'RAM available: {mem.available / 1e9:.1f} GB')

DEVICE = torch.device('cuda:0' if torch.cuda.is_available() else 'cpu')
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print(f'GPU: {props.name}  ({props.total_memory/1e9:.1f} GB)')
print(f'Active device: {DEVICE}')

OUT = '/content/drive/MyDrive/DAMF/logs'
os.makedirs(OUT, exist_ok=True)

total, used, free = _shutil.disk_usage('/content/drive/MyDrive')
print(f'Drive free space: {free / 1e9:.2f} GB')
if free / 1e9 < 3.0:
    print('!! WARNING: low Drive space. Clean up before launching training.')

SEEDS = [42, 0, 123]
CFG = {
    'batch_size': 16, 'beam_size': 3, 'num_workers': 1,  # 1 worker, per Notebook 6 fix
    'total_epochs': 5, 'lr': 1e-4,  # matches original text-only LoRA baseline's lr_lora
    'weight_decay': 0.01,
    'lora_r': 16, 'lora_alpha': 32, 'lora_dropout': 0.05,
    'rt_log_every_n_steps': 10, 'rt_epsilon': 1e-8,
}
print(f'\nCFG loaded. Output: {OUT}')

## 4. Shared metric/eval utilities (memory-optimized, per Notebook 6)

In [ ]:
def compute_bleu4(predictions, references):
    smoother = SmoothingFunction().method4
    pred_tokens = [p.split() for p in predictions]
    ref_tokens = [[r.split() for r in refs] for refs in references]
    return corpus_bleu(ref_tokens, pred_tokens, smoothing_function=smoother)

from pycocoevalcap.cider.cider import Cider
from pycocoevalcap.tokenizer.ptbtokenizer import PTBTokenizer
import evaluate as hf_evaluate

_CIDER_SCORER = Cider()
_PTB_TOKENIZER = PTBTokenizer()
_METEOR_METRIC = hf_evaluate.load('meteor')

def compute_cider(predictions, references):
    try:
        gts = {i: [{'caption': r} for r in refs] for i, refs in enumerate(references)}
        res = {i: [{'caption': p}] for i, p in enumerate(predictions)}
        sc, _ = _CIDER_SCORER.compute_score(
            _PTB_TOKENIZER.tokenize(gts), _PTB_TOKENIZER.tokenize(res))
        return float(sc)
    except Exception as e:
        print(f'  CIDEr error: {e}')
        return None

def compute_meteor(predictions, references):
    try:
        flat_refs = [refs[0] for refs in references]
        result = _METEOR_METRIC.compute(predictions=predictions, references=flat_refs)
        return float(result['meteor'])
    except Exception as e:
        print(f'  METEOR error: {e}')
        return None

def get_val_loss(model, loader):
    model.eval()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in loader:
            inputs = {
                'pixel_values': batch['pixel_values'].to(DEVICE),
                'input_ids': batch['input_ids'].to(DEVICE),
                'attention_mask': batch['attention_mask'].to(DEVICE),
            }
            inputs['labels'] = inputs['input_ids'].clone()
            with torch.amp.autocast('cuda'):
                total += model(**inputs).loss.item()
            n += 1
    return total / max(n, 1)

def train_one_epoch_lora(model, loader, optimizer, scaler, step_counter=None):
    """Standard training step -- no gradient rescaling. Rt is still
    logged for comparability with every other method's rt_log field.
    Uses the shared GradientTracker convention (language/visual ratio).
    """
    model.train()
    total_loss, n_batches, rt_log = 0.0, 0, []
    if step_counter is None:
        step_counter = [0]
    for batch in loader:
        inputs = {
            'pixel_values': batch['pixel_values'].to(DEVICE),
            'input_ids': batch['input_ids'].to(DEVICE),
            'attention_mask': batch['attention_mask'].to(DEVICE),
        }
        inputs['labels'] = inputs['input_ids'].clone()
        optimizer.zero_grad()
        with torch.amp.autocast('cuda'):
            loss = model(**inputs).loss
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        total_loss += loss.item()
        n_batches += 1
        step_counter[0] += 1
    return total_loss / max(n_batches, 1), rt_log

def seed_everything(seed):
    random.seed(seed); np.random.seed(seed)
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def worker_init_fn(worker_id):
    worker_seed = torch.initial_seed() % (2**32)
    np.random.seed(worker_seed); random.seed(worker_seed)

def save_logs(logs, filename):
    path = os.path.join(OUT, filename)
    with open(path, 'w') as f:
        json.dump(logs, f, indent=2)
    print(f'  Saved: {filename}')
    return path

def check_disk_space(min_gb=2.0):
    free_gb = _shutil.disk_usage(OUT).free / 1e9
    if free_gb < min_gb:
        print(f'  !! DISK WARNING: only {free_gb:.2f} GB free.')
        return False
    return True

def save_checkpoint(model, filename):
    local_path = os.path.join('/content', filename)
    try:
        torch.save(model.state_dict(), local_path)
        return local_path
    except RuntimeError as e:
        print(f'  !! Local checkpoint save failed ({filename}): {e}')
        return None

def sync_checkpoint_to_drive(filename):
    if not check_disk_space():
        return
    local_path = os.path.join('/content', filename)
    drive_path = os.path.join(OUT, filename)
    if os.path.exists(local_path):
        _shutil.copy(local_path, drive_path)

def delete_checkpoint(filename):
    local_path = os.path.join('/content', filename)
    drive_path = os.path.join(OUT, filename)
    if os.path.exists(local_path):
        os.remove(local_path)
    if os.path.exists(drive_path):
        os.remove(drive_path)

BLIP_PROCESSOR = BlipProcessor.from_pretrained('Salesforce/blip-image-captioning-base')
print('Shared utilities defined.')

## 5. Load base model for module inspection (do not skip)

In [ ]:
_inspect_model = BlipForConditionalGeneration.from_pretrained(
    'Salesforce/blip-image-captioning-base')
print('Model loaded for inspection.')

## 6. DIAGNOSTIC -- inspect actual module names before choosing target_modules

**Run this cell and read its output before proceeding.** It lists every
`nn.Linear` layer inside `vision_model` and `text_decoder`, so we can see
the REAL attention module names rather than assuming. This is exactly
the step that was skipped when the original "LoRA (text-only)" baseline
silently missed the visual encoder.

In [ ]:
import torch.nn as nn

vision_linears = []
text_linears = []
for name, module in _inspect_model.named_modules():
    if isinstance(module, nn.Linear):
        if 'vision_model' in name:
            vision_linears.append(name)
        elif 'text_decoder' in name:
            text_linears.append(name)

print(f'Found {len(vision_linears)} Linear layers under vision_model.')
print(f'Found {len(text_linears)} Linear layers under text_decoder.\n')

print('Sample of vision_model Linear layer names (first 15):')
for n in vision_linears[:15]:
    print(f'  {n}')

print('\nSample of text_decoder Linear layer names (first 15):')
for n in text_linears[:15]:
    print(f'  {n}')

# Extract the distinct final-component names (what target_modules matches against)
vision_leaf_names = sorted(set(n.split('.')[-1] for n in vision_linears))
text_leaf_names = sorted(set(n.split('.')[-1] for n in text_linears))
print(f'\nDistinct vision_model leaf module names: {vision_leaf_names}')
print(f'Distinct text_decoder leaf module names: {text_leaf_names}')

print('\n' + '='*70)
print('READ THIS: peft target_modules matches against the LEAF name')
print('(the last component after the final "."), via substring or exact')
print('match depending on peft version. The original text-only LoRA')
print('baseline used target_modules=["query","value"], which matches')
print(f'against text_decoder leaf names {text_leaf_names} but likely')
print(f'does NOT match vision_model leaf names {vision_leaf_names}')
print('(BLIP\'s ViT typically fuses QKV into one Linear, e.g. "qkv",')
print('rather than separate query/key/value Linears like the text side).')
print('='*70)

## 7. Set MULTIMODAL_TARGET_MODULES based on Cell 6's actual output

**Edit this cell manually** after reading Cell 6's printed leaf-name
lists. The list below is a reasonable starting guess based on standard
BLIP/ViT architecture, but VERIFY against your own Cell 6 output before
trusting it -- this is the whole point of the diagnostic step.

In [ ]:
# EDIT if Cell 6's output shows different leaf names than assumed here.
# Guess based on standard BLIP architecture:
#   - text_decoder attention uses separate 'query' / 'key' / 'value' Linears
#   - vision_model (ViT) attention typically fuses QKV into one 'qkv' Linear,
#     with a separate output-projection Linear often named 'projection'
MULTIMODAL_TARGET_MODULES = ['query', 'value', 'qkv', 'projection']

# Verify this list actually produces >0 matches in BOTH vision_model and
# text_decoder before committing to training. This mirrors exactly the
# audit that was missing the first time.
test_config = LoraConfig(
    r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
    target_modules=MULTIMODAL_TARGET_MODULES,
    lora_dropout=CFG['lora_dropout'], bias='none',
)
test_peft_model = get_peft_model(_inspect_model, test_config)

n_vis_lora, n_lang_lora = 0, 0
for name, param in test_peft_model.named_parameters():
    if param.requires_grad:
        if 'vision_model' in name:
            n_vis_lora += 1
        elif 'text_decoder' in name:
            n_lang_lora += 1

print(f'With target_modules={MULTIMODAL_TARGET_MODULES}:')
print(f'  Trainable LoRA params in vision_model: {n_vis_lora}')
print(f'  Trainable LoRA params in text_decoder: {n_lang_lora}')

if n_vis_lora == 0:
    print('\n!! FAILURE: still 0 visual parameters. Go back to Cell 6\'s')
    print('output, find the actual vision_model leaf names, and add them')
    print('to MULTIMODAL_TARGET_MODULES above. Do NOT proceed to training')
    print('until this shows a nonzero visual count.')
else:
    print('\nOK: both pathways have trainable LoRA parameters. Proceed.')

# Clean up the test model
del test_peft_model, _inspect_model
gc.collect()
torch.cuda.empty_cache()

---
# PART A -- UICD
---

## A1. Download UICD from Kaggle

In [ ]:
kaggle_json_src = Path('/content/drive/MyDrive/kaggle.json')
assert kaggle_json_src.exists(), 'kaggle.json not found in Drive.'
os.makedirs('/root/.kaggle', exist_ok=True)
_shutil.copy(kaggle_json_src, '/root/.kaggle/kaggle.json')
os.chmod('/root/.kaggle/kaggle.json', 0o600)

uicd_dir = Path('/content/uicd_data')
if not uicd_dir.exists() or not any(uicd_dir.iterdir()):
    uicd_dir.mkdir(exist_ok=True)
    subprocess.run(['kaggle', 'datasets', 'download',
                    '-d', 'kiranmuhammad/uicd-underwater-dataset',
                    '-p', str(uicd_dir), '--unzip'], check=True)

captions_files = list(uicd_dir.rglob('UIC-captions.txt'))
assert captions_files, 'UIC-captions.txt not found.'
UICD_CAPS = str(captions_files[0])
image_dirs = [d for d in captions_files[0].parent.iterdir()
             if d.is_dir() and 'image' in d.name.lower()]
UICD_IMAGES = str(image_dirs[0])
print(f'UICD ready. Images: {len(list(Path(UICD_IMAGES).glob("*")))}')

## A2. UICD dataset + loaders

In [ ]:
def load_uicd_captions(captions_path):
    image_captions = defaultdict(list)
    with open(captions_path, 'r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            parts = line.split('#')
            if len(parts) < 2:
                continue
            img_name = parts[0].strip()
            cap_part = parts[1].strip()
            caption = cap_part.split(' ', 1)[1].strip() if ' ' in cap_part else cap_part
            if img_name and caption:
                image_captions[img_name].append(caption)
    return dict(image_captions)

uicd_image_captions = load_uicd_captions(UICD_CAPS)
_rng = random.Random(42)
_all = sorted(uicd_image_captions.keys())
_rng.shuffle(_all)
n = len(_all)
tr_end = int(0.70 * n)
va_end = int(0.85 * n)
UICD_SPLITS = {'train': _all[:tr_end], 'val': _all[tr_end:va_end], 'test': _all[va_end:]}
print(f"UICD split: train={len(UICD_SPLITS['train'])} val={len(UICD_SPLITS['val'])} "
      f"test={len(UICD_SPLITS['test'])}")

class UICDataset(Dataset):
    def __init__(self, image_list, image_captions, image_folder, processor,
                max_length=30, deterministic=False):
        self.image_list = image_list
        self.image_captions = image_captions
        self.image_folder = image_folder
        self.processor = processor
        self.max_length = max_length
        self.deterministic = deterministic

    def __len__(self):
        return len(self.image_list)

    def __getitem__(self, idx):
        img_name = self.image_list[idx]
        image = Image.open(os.path.join(self.image_folder, img_name)).convert('RGB')
        caption = (self.image_captions[img_name][0] if self.deterministic
                  else random.choice(self.image_captions[img_name]))
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'image_name': img_name,
        }

g = torch.Generator(); g.manual_seed(0)
uicd_train_loader = DataLoader(
    UICDataset(UICD_SPLITS['train'], uicd_image_captions, UICD_IMAGES, BLIP_PROCESSOR,
              deterministic=False),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
    pin_memory=True, worker_init_fn=worker_init_fn, generator=g)
uicd_val_loader = DataLoader(
    UICDataset(UICD_SPLITS['val'], uicd_image_captions, UICD_IMAGES, BLIP_PROCESSOR,
              deterministic=True),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True)
print(f'UICD train batches: {len(uicd_train_loader)}  val: {len(uicd_val_loader)}')

@torch.no_grad()
def evaluate_blip_uicd(model, loader, processor, image_captions, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(pixel_values=batch['pixel_values'].to(DEVICE),
                                 max_length=30, num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        for name in batch['image_name']:
            references.append(image_captions[name])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

## A3. UICD Multimodal LoRA runner

In [ ]:
def run_uicd_multimodal_lora(seed):
    exp_name = f'multimodal_lora_uicd_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= CFG['total_epochs']:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'UICD', 'seed': seed,
            'method': 'multimodal_lora', 'lr': CFG['lr'], 'epochs': CFG['total_epochs'],
            'target_modules': MULTIMODAL_TARGET_MODULES,
            'lora_r': CFG['lora_r'], 'lora_alpha': CFG['lora_alpha'],
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [], 'rt_log': [],
        }
        done = 0

    seed_everything(seed)
    base_model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base')
    lora_config = LoraConfig(
        r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
        target_modules=MULTIMODAL_TARGET_MODULES,
        lora_dropout=CFG['lora_dropout'], bias='none',
    )
    model = get_peft_model(base_model, lora_config).to(DEVICE)

    if 'n_visual_lora_params' not in logs:
        n_vis, n_lang = 0, 0
        for name, p in model.named_parameters():
            if p.requires_grad:
                if 'vision_model' in name:
                    n_vis += 1
                elif 'text_decoder' in name:
                    n_lang += 1
        logs['n_visual_lora_params'] = n_vis
        logs['n_language_lora_params'] = n_lang
        print(f'  [{exp_name}] LoRA modules -- visual: {n_vis}  language: {n_lang}')

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    step_counter = [done * len(uicd_train_loader)]
    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0

    for epoch in range(done + 1, CFG['total_epochs'] + 1):
        avg_loss, rt_log = train_one_epoch_lora(
            model, uicd_train_loader, optimizer, scaler, step_counter=step_counter)
        val_loss = get_val_loss(model, uicd_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_uicd(
            model, uicd_val_loader, BLIP_PROCESSOR, uicd_image_captions, CFG)

        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)

        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = [r[:2] for r in refs[:20]]

        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f"  [{exp_name}] epoch {epoch}/{CFG['total_epochs']} | "
              f'train={avg_loss:.4f} val={val_loss:.4f} BLEU-4={bleu4:.4f} '
              f'CIDEr={cs} METEOR={ms}')

        save_checkpoint(model, f'{exp_name}_last.pt')
        if epoch % 2 == 0 or epoch == CFG['total_epochs']:
            sync_checkpoint_to_drive(f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')
        gc.collect()
        torch.cuda.empty_cache()

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_uicd_multimodal_lora defined.')

## A4. LAUNCH -- UICD, all 3 seeds

In [ ]:
print('=' * 60)
print('MULTIMODAL LoRA -- UICD, all 3 seeds')
print('=' * 60)
uicd_results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    uicd_results[seed] = run_uicd_multimodal_lora(seed)

print('\nUICD ALL SEEDS COMPLETE')
for seed, logs in uicd_results.items():
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}  "
          f"visual_lora_params={logs.get('n_visual_lora_params')}  "
          f"language_lora_params={logs.get('n_language_lora_params')}")

---
# PART B -- RSICD
---

## B1. RSICD dataset + loaders

In [ ]:
print('Loading RSICD from HuggingFace...')
rsicd_raw = load_dataset('arampacha/rsicd')
print(f"RSICD: {len(rsicd_raw['train'])} train / {len(rsicd_raw['valid'])} val")

class RSICDDataset(Dataset):
    def __init__(self, hf_split, processor, max_length=30, deterministic=False):
        self.data = hf_split
        self.processor = processor
        self.max_length = max_length
        self.deterministic = deterministic

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        caption = (item['captions'][0] if self.deterministic
                  else random.choice(item['captions']))
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'captions': item['captions'], 'filename': item['filename'],
        }

def rsicd_collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'captions': [b['captions'] for b in batch],
        'filename': [b['filename'] for b in batch],
    }

rsicd_train_loader = DataLoader(
    RSICDDataset(rsicd_raw['train'], BLIP_PROCESSOR, deterministic=False),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rsicd_collate, worker_init_fn=worker_init_fn)
rsicd_val_loader = DataLoader(
    RSICDDataset(rsicd_raw['valid'], BLIP_PROCESSOR, deterministic=True),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rsicd_collate)
print(f'RSICD train batches: {len(rsicd_train_loader)}  val: {len(rsicd_val_loader)}')

@torch.no_grad()
def evaluate_blip_rsicd(model, loader, processor, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(pixel_values=batch['pixel_values'].to(DEVICE),
                                 max_length=30, num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        references.extend(batch['captions'])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

## B2. RSICD Multimodal LoRA runner

In [ ]:
def run_rsicd_multimodal_lora(seed):
    exp_name = f'multimodal_lora_rsicd_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= CFG['total_epochs']:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'RSICD', 'seed': seed,
            'method': 'multimodal_lora', 'lr': CFG['lr'], 'epochs': CFG['total_epochs'],
            'target_modules': MULTIMODAL_TARGET_MODULES,
            'lora_r': CFG['lora_r'], 'lora_alpha': CFG['lora_alpha'],
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [], 'rt_log': [],
        }
        done = 0

    seed_everything(seed)
    base_model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base')
    lora_config = LoraConfig(
        r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
        target_modules=MULTIMODAL_TARGET_MODULES,
        lora_dropout=CFG['lora_dropout'], bias='none',
    )
    model = get_peft_model(base_model, lora_config).to(DEVICE)

    if 'n_visual_lora_params' not in logs:
        n_vis, n_lang = 0, 0
        for name, p in model.named_parameters():
            if p.requires_grad:
                if 'vision_model' in name:
                    n_vis += 1
                elif 'text_decoder' in name:
                    n_lang += 1
        logs['n_visual_lora_params'] = n_vis
        logs['n_language_lora_params'] = n_lang
        print(f'  [{exp_name}] LoRA modules -- visual: {n_vis}  language: {n_lang}')

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    step_counter = [done * len(rsicd_train_loader)]
    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0

    for epoch in range(done + 1, CFG['total_epochs'] + 1):
        avg_loss, rt_log = train_one_epoch_lora(
            model, rsicd_train_loader, optimizer, scaler, step_counter=step_counter)
        val_loss = get_val_loss(model, rsicd_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_rsicd(
            model, rsicd_val_loader, BLIP_PROCESSOR, CFG)

        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)

        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = [r[:2] for r in refs[:20]]

        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f"  [{exp_name}] epoch {epoch}/{CFG['total_epochs']} | "
              f'train={avg_loss:.4f} val={val_loss:.4f} BLEU-4={bleu4:.4f} '
              f'CIDEr={cs} METEOR={ms}')

        save_checkpoint(model, f'{exp_name}_last.pt')
        if epoch % 2 == 0 or epoch == CFG['total_epochs']:
            sync_checkpoint_to_drive(f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')
        gc.collect()
        torch.cuda.empty_cache()

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_rsicd_multimodal_lora defined.')

## B3. LAUNCH -- RSICD, all 3 seeds

In [ ]:
print('=' * 60)
print('MULTIMODAL LoRA -- RSICD, all 3 seeds')
print('=' * 60)
rsicd_results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    rsicd_results[seed] = run_rsicd_multimodal_lora(seed)

print('\nRSICD ALL SEEDS COMPLETE')
for seed, logs in rsicd_results.items():
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}")

---
# PART C -- ROCOv2
---

## C1. ROCOv2 dataset + loaders

In [ ]:
print('Loading ROCOv2 (full dataset) from HuggingFace...')
rocov2_raw = load_dataset('eltorio/ROCOv2-radiology')
print(f"ROCOv2: {len(rocov2_raw['train'])} train / {len(rocov2_raw['validation'])} val")

ROCOV2_MAX_LENGTH = 40

class ROCOv2Dataset(Dataset):
    def __init__(self, hf_split, processor, max_length):
        self.data = hf_split
        self.processor = processor
        self.max_length = max_length

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]
        image = item['image'].convert('RGB')
        caption = item['caption']
        inputs = self.processor(images=image, text=caption, return_tensors='pt',
                                padding='max_length', truncation=True,
                                max_length=self.max_length)
        return {
            'pixel_values': inputs['pixel_values'].squeeze(0),
            'input_ids': inputs['input_ids'].squeeze(0),
            'attention_mask': inputs['attention_mask'].squeeze(0),
            'caption': caption, 'image_id': item['image_id'],
        }

def rocov2_collate(batch):
    return {
        'pixel_values': torch.stack([b['pixel_values'] for b in batch]),
        'input_ids': torch.stack([b['input_ids'] for b in batch]),
        'attention_mask': torch.stack([b['attention_mask'] for b in batch]),
        'captions': [[b['caption']] for b in batch],
        'image_id': [b['image_id'] for b in batch],
    }

rocov2_train_loader = DataLoader(
    ROCOv2Dataset(rocov2_raw['train'], BLIP_PROCESSOR, ROCOV2_MAX_LENGTH),
    batch_size=CFG['batch_size'], shuffle=True, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rocov2_collate, worker_init_fn=worker_init_fn)
rocov2_val_loader = DataLoader(
    ROCOv2Dataset(rocov2_raw['validation'], BLIP_PROCESSOR, ROCOV2_MAX_LENGTH),
    batch_size=CFG['batch_size'], shuffle=False, num_workers=CFG['num_workers'],
    pin_memory=True, collate_fn=rocov2_collate)
print(f'ROCOv2 train batches: {len(rocov2_train_loader)}  val: {len(rocov2_val_loader)}')

@torch.no_grad()
def evaluate_blip_rocov2(model, loader, processor, cfg):
    model.eval()
    predictions, references = [], []
    for batch in loader:
        gen_ids = model.generate(pixel_values=batch['pixel_values'].to(DEVICE),
                                 max_length=ROCOV2_MAX_LENGTH, num_beams=cfg['beam_size'])
        predictions.extend(processor.batch_decode(gen_ids, skip_special_tokens=True))
        references.extend(batch['captions'])
    bleu4 = compute_bleu4(predictions, references)
    cider = compute_cider(predictions, references)
    meteor = compute_meteor(predictions, references)
    return bleu4, cider, meteor, predictions, references

## C2. ROCOv2 Multimodal LoRA runner

In [ ]:
def run_rocov2_multimodal_lora(seed):
    exp_name = f'multimodal_lora_rocov2_seed{seed}'
    json_path = os.path.join(OUT, f'{exp_name}.json')

    if os.path.exists(json_path):
        with open(json_path) as f:
            logs = json.load(f)
        done = len(logs['bleu4_per_epoch'])
        if done >= CFG['total_epochs']:
            print(f'  [{exp_name}] already complete. Skipping.')
            return logs
    else:
        logs = {
            'experiment': exp_name, 'dataset': 'ROCOv2', 'seed': seed,
            'method': 'multimodal_lora', 'lr': CFG['lr'], 'epochs': CFG['total_epochs'],
            'target_modules': MULTIMODAL_TARGET_MODULES,
            'lora_r': CFG['lora_r'], 'lora_alpha': CFG['lora_alpha'],
            'bleu4_per_epoch': [], 'cider_per_epoch': [], 'meteor_per_epoch': [],
            'train_loss_per_epoch': [], 'val_loss_per_epoch': [], 'rt_log': [],
        }
        done = 0

    seed_everything(seed)
    base_model = BlipForConditionalGeneration.from_pretrained(
        'Salesforce/blip-image-captioning-base')
    lora_config = LoraConfig(
        r=CFG['lora_r'], lora_alpha=CFG['lora_alpha'],
        target_modules=MULTIMODAL_TARGET_MODULES,
        lora_dropout=CFG['lora_dropout'], bias='none',
    )
    model = get_peft_model(base_model, lora_config).to(DEVICE)

    if 'n_visual_lora_params' not in logs:
        n_vis, n_lang = 0, 0
        for name, p in model.named_parameters():
            if p.requires_grad:
                if 'vision_model' in name:
                    n_vis += 1
                elif 'text_decoder' in name:
                    n_lang += 1
        logs['n_visual_lora_params'] = n_vis
        logs['n_language_lora_params'] = n_lang
        print(f'  [{exp_name}] LoRA modules -- visual: {n_vis}  language: {n_lang}')

    ckpt_path = os.path.join(OUT, f'{exp_name}_last.pt')
    if done > 0 and os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        print(f'  [{exp_name}] loaded checkpoint from epoch {done}')

    optimizer = AdamW(
        filter(lambda p: p.requires_grad, model.parameters()),
        lr=CFG['lr'], weight_decay=CFG['weight_decay'])
    scaler = torch.amp.GradScaler('cuda')
    step_counter = [done * len(rocov2_train_loader)]
    best_bleu4 = max(logs['bleu4_per_epoch']) if logs['bleu4_per_epoch'] else 0.0

    for epoch in range(done + 1, CFG['total_epochs'] + 1):
        avg_loss, rt_log = train_one_epoch_lora(
            model, rocov2_train_loader, optimizer, scaler, step_counter=step_counter)
        val_loss = get_val_loss(model, rocov2_val_loader)
        bleu4, cider, meteor, preds, refs = evaluate_blip_rocov2(
            model, rocov2_val_loader, BLIP_PROCESSOR, CFG)

        logs['train_loss_per_epoch'].append(round(avg_loss, 4))
        logs['val_loss_per_epoch'].append(round(val_loss, 4))
        logs['bleu4_per_epoch'].append(round(bleu4, 4))
        logs['cider_per_epoch'].append(round(cider, 4) if cider else None)
        logs['meteor_per_epoch'].append(round(meteor, 4) if meteor else None)

        if bleu4 > best_bleu4:
            best_bleu4 = bleu4
            logs['best_bleu4'] = bleu4
            logs['best_cider'] = cider
            logs['best_meteor'] = meteor
            logs['best_epoch'] = epoch
            logs['best_predictions'] = preds[:20]
            logs['best_references'] = refs[:20]

        cs = f'{cider:.4f}' if cider else 'N/A'
        ms = f'{meteor:.4f}' if meteor else 'N/A'
        print(f"  [{exp_name}] epoch {epoch}/{CFG['total_epochs']} | "
              f'train={avg_loss:.4f} val={val_loss:.4f} BLEU-4={bleu4:.4f} '
              f'CIDEr={cs} METEOR={ms}')

        save_checkpoint(model, f'{exp_name}_last.pt')
        if epoch % 2 == 0 or epoch == CFG['total_epochs']:
            sync_checkpoint_to_drive(f'{exp_name}_last.pt')
        save_logs(logs, f'{exp_name}.json')
        gc.collect()
        torch.cuda.empty_cache()

    del model, base_model
    gc.collect()
    torch.cuda.empty_cache()
    delete_checkpoint(f'{exp_name}_last.pt')
    print(f'  [{exp_name}] complete. Best BLEU-4: {best_bleu4:.4f}\n')
    return logs

print('run_rocov2_multimodal_lora defined.')

## C3. LAUNCH -- ROCOv2, all 3 seeds

In [ ]:
print('=' * 60)
print('MULTIMODAL LoRA -- ROCOv2, all 3 seeds')
print('=' * 60)
rocov2_results = {}
for seed in SEEDS:
    print(f"\n{'#'*60}\n# SEED {seed}\n{'#'*60}")
    rocov2_results[seed] = run_rocov2_multimodal_lora(seed)

print('\nROCOv2 ALL SEEDS COMPLETE')
for seed, logs in rocov2_results.items():
    print(f"seed {seed}: best_bleu4={logs.get('best_bleu4'):.4f}")

print('\n' + '=' * 60)
print('ALL THREE DATASETS COMPLETE FOR MULTIMODAL LoRA')
print('ALL PLANNED EXPERIMENTS NOW COMPLETE')
print('=' * 60)